# Pasado `gpu-hybrid`: CUDA potvrda automatskog izbora `gelsd` / eager A2

Pre pokretanja: **Runtime > Change runtime type > GPU (A100)**, zatim **Runtime > Disconnect and delete runtime**, pa **Runtime > Run all**.

Pokreće samo `experiments/test_auto_plane_fit.py --device cuda` na commitu `1d0c49b`. Na CUDA-i se skup uređaja za automatski izbor ne proširuje, pa test proverava stvarno podešavanje: pragovi 512 (`float64`) i 640 (`float32`), granični slučajevi, override, nedozvoljene kombinacije prekidača, batched prolazi naspram eksplicitnog puta i sekvencijalne petlje, chunking, da CPU ostaje na `gelsd`-u i da se ne kompajlira nijedan Inductor graf.

Trajanje: par minuta (procena). Kad završi, pošalji `auto_plane_fit_cuda.zip`.

In [ ]:
import subprocess
import torch

subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "Uključi GPU: Runtime > Change runtime type > GPU"
GPU = torch.cuda.get_device_name(0)
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda, "| GPU:", GPU)
assert "A100" in GPU, f"Pragovi su izmereni na A100, a ovo je {GPU}."

In [ ]:
import os
import shutil

REPO = "/content/Pasado"
URL = "https://github.com/dimitrijepesic/Pasado.git"
COMMIT = "1d0c49b"

os.chdir("/content")
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "-c", "http.version=HTTP/1.1", "clone", "--branch", "gpu-hybrid",
                "--single-branch", URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", COMMIT], check=True)
head = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
assert head.startswith(COMMIT), f"HEAD je {head}, a ocekivan je {COMMIT}"
print("HEAD:", head)
os.chdir(REPO)
OUT = "/content/auto_plane_fit_cuda"
os.makedirs(OUT, exist_ok=True)

In [ ]:
%pip install -q scikit-learn mpmath

In [ ]:
cmd = f'''
set -o pipefail
PYTHONUNBUFFERED=1 python experiments/test_auto_plane_fit.py --device cuda 2>&1 | tee {OUT}/test_auto_plane_fit_cuda.log
'''
rc = subprocess.run(["bash", "-lc", cmd], cwd=REPO).returncode
print("exit code:", rc)

In [ ]:
import json
from zipfile import ZIP_DEFLATED, ZipFile
from google.colab import files

with open(f"{OUT}/colab_environment.json", "w") as f:
    json.dump(dict(gpu=GPU, vram_gib=torch.cuda.get_device_properties(0).total_memory / 2**30,
                   torch=torch.__version__, cuda=torch.version.cuda, commit=head, exit_code=rc), f, indent=2)
archive = "/content/auto_plane_fit_cuda.zip"
with ZipFile(archive, "w", ZIP_DEFLATED) as zf:
    for name in sorted(os.listdir(OUT)):
        zf.write(os.path.join(OUT, name), os.path.join("auto_plane_fit_cuda", name))
files.download(archive)
assert rc == 0, "test nije prošao; pošalji zip"